# Compute a covariance analogous to DES × Planck

The supplied file has 1,809 entries, but this generator currently computes only its first 1,500 galaxy/shear entries. Their 635 retained entries form the supported 3×2pt submatrix. CMB lensing auto- and cross-covariances are not generated. The supplied full joint matrix remains available to the likelihood.

This notebook computes Gaussian (G), super-sample (SSC), connected non-Gaussian
(cNG), and total covariance separately. It then reads the project's supplied
matrix and applies **the same likelihood mask** to both. The comparison concerns
analogous measurements; it is not a reproduction of the supplied covariance.
No likelihood data file is overwritten and no covariance is installed in C.

| Measurement choice | Value |
| --- | --- |
| Dataset | [data/Y3xPlanckPR4.dataset](data/Y3xPlanckPR4.dataset) |
| Lens / source bins | 6 / 4 |
| Bins per two-point observable | 30, 0.25–250 arcmin |
| Generated entries before cuts | 1,500 |
| Entries after the selected likelihood mask | 635 |

The forecast uses massless neutrinos, linear galaxy bias, zero
magnification/RSD and a spherical-cap footprint.

Gaussian galaxy–galaxy and galaxy–shear spectra include non-Limber
corrections; shear–shear remains Limber. Gaussian NLA/TATT is optional below.

SSC/cNG retain their zero-IA Limber model. SSC uses the isotropic halo
response and cNG the halo trispectrum. Massive-neutrino non-Gaussian terms
are not implemented.

These physical choices differ from the supplied likelihood matrices.
Matching their measurement layout does not establish physical or numerical
equivalence.



The [covariance guide](covariance/README.md) explains survey inputs and how to
start Jupyter. The decomposition follows
[Krause & Eifler, Appendix A](https://arxiv.org/abs/1601.05779) and
[Takada & Hu](https://arxiv.org/abs/1302.6994).
Covariance generation is excluded from the default build. Follow the
[project build steps](README.md#computing_covariances): after activating
Cocoa, unset `IGNORE_COSMOLIKE_DESXPLANCK_COVARIANCE` and recompile
this project. Restart the notebook kernel after rebuilding.


In [ ]:
import os
from pathlib import Path
import sys

# BLAS stays serial; the compiled CosmoLike loops own the OpenMP workers.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
root = Path(os.environ["ROOTDIR"])
project = root/"projects/desy1xplanck"
sys.path.insert(0, str(root/"external_modules/code/cosmolike_core"))
sys.path.insert(0, str(root/"external_modules/code/CAMB"))
sys.path.insert(0, str(project/"interface"))
sys.path.insert(0, str(project/"covariance"))

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from threadpoolctl import threadpool_limits
import cosmolike_desy1xplanck_interface as ci
import desy1xplanck_covariance as survey
from cosmolike_notebook_utils import covariance as cov
from cosmolike_notebook_utils import plot_covariances as pcov
from cosmolike_notebook_utils.covariance.likelihood import (
    read_likelihood_covariance,
    select_likelihood_entries,
)
from cosmolike_notebook_utils.covariance.forecast import save_forecast

blas_limit = threadpool_limits(limits=1, user_api="blas")
ci.set_omp_threads(n=8)
matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["font.family"] = "STIXGeneral"

## Choose the measurement and numerical settings

The default run computes the project's native measurement once at boost 1.
Set `boosts = [1, 2]` to add the numerical comparison below. For galaxy/shear
forecasts, `spaces = ["real", "fourier"]` computes both transformations; the
companion space is illustrative and is not compared to the supplied matrix.
The cluster generator returns the joint real-space measurement only.

`accuracy_boost` multiplies the internal interpolation refinements and
multipole cutoffs in [covariance/default.yaml](covariance/default.yaml).
Doubled grids retain old nodes. `integration_accuracy` independently selects
96, 128, 256, 512 or 1,024 precomputed GSL nodes per panel at levels 0–4.
Keep the scientific bins, cosmology and CAMB inputs fixed when refining.

The defaults remain **candidate accuracy settings**. Positive definiteness
and a small change in diagonal errors do not establish convergence of all
covariance modes or of Fisher parameter errors. A comparison with the supplied
matrix also mixes physical choices with numerical differences.

The `gaussian` mapping selects the Gaussian spectra separately from numerical
accuracy. Use `ia="NLA", A1=0.6` for an illustrative constant NLA amplitude,
or `ia="TATT", A1=0.6, A2=0.2, B_TA=0.5` for TATT. A list gives one amplitude
per source bin. TATT B modes enter real-space shear covariance. These choices
do not change SSC/cNG or their original mean-subtraction signal.


In [ ]:
boosts = [1]
spaces = ['real']
gaussian = {"nonlimber": True, "ia": "none"}
settings = survey.configuration(
    accuracy_boost=boosts[0], gaussian=gaussian,
)
dataset = project/"data/Y3xPlanckPR4.dataset"

# These are physical assumptions, not inferred from normalized n(z) columns.
for key in (
    "area_deg2",
    "lens_density_arcmin2",
    "source_density_arcmin2",
    "sigma_e_component",
    "bias",
    "cosmology",
    "accuracy_parameters",
    "gaussian",
):
    print(key, settings[key])

camb_tables = survey.initialize(interface=ci, settings=settings)
ci.set_omp_threads(n=8)


def progress(stage, elapsed):
    """Report completed stages of this calculation, in seconds."""
    print(f"{elapsed:8.2f} s  {stage}")

## Compute G, SSC, cNG and their sum

G describes covariance made from pairs of two-point spectra, including
catalog shot noise and shape noise. SSC couples measurements through a
background density fluctuation larger than the footprint. cNG is the remaining
connected four-point contribution in the adopted halo model.

Angular bins are consecutive within each observable. The Fourier companion
uses one E-mode row per source pair, with integer multipoles weighted by
$2\ell+1$ inside each band. The Fourier likelihood evaluates its mean spectrum
at logarithmic band centers; the generated covariance instead averages integer
modes within bands. It is therefore an analogous bandpower calculation.

Each computation retains internal cross-bin spectra even when the corresponding
observable is absent from the measured vector. The loop below changes only
numerical resolution, and stores all physical components at each requested boost.

In [ ]:
# Each space uses its own projection. Changing the boost refines tables
# while preserving cosmology, observable ordering and measured bins.
results = {}
for space in spaces:
    if space not in ['real', 'fourier']:
        raise ValueError("Choose a measurement space supported by this adapter")
    results[space] = {}
    for boost in boosts:
        refined = dict(settings)
        refined.update(cov.covariance_accuracy(
            accuracy_boost=boost, **settings["accuracy_parameters"],
        ))
        print(space, "accuracy boost", boost)
        results[space][boost] = survey.compute(
            interface=ci, settings=refined, space=space, progress=progress,
        )

native_space = 'real'
forecast = results[native_space][boosts[-1]]
print("Full covariance shape:", forecast["total"].shape)

## Apply the project's likelihood cuts

The supplied mask selects measurements, so the same selection must act on
both axes of G, SSC, cNG and their sum. The saved `indices` preserve the original
file positions after compaction. Probe labels refer to contiguous groups in the
uncut likelihood vector, including any measured-pair exclusions.

The reader adds the Gaussian and combined non-Gaussian columns of legacy
CosmoCov files. It does not guess how that combined column divides into SSC
and cNG. The supplied total appears in the upper triangle of the plot and
the newly computed total in the lower triangle. Each uses its own diagonal
for correlation normalization; differences here are **not** a numerical
convergence test.

In [ ]:
supplied = read_likelihood_covariance(dataset=dataset, size=1500)
block_sizes = [300, 300, 720, 180]
block_labels = ['$\\xi_+$', '$\\xi_-$', '$\\gamma_t$', '$w$']
selected = select_likelihood_entries(
    forecast=forecast,
    supplied=supplied,
    block_sizes=block_sizes,
    block_labels=block_labels,
)
print("Supplied file size:", supplied["file_size"])
print("Retained comparison size:", len(selected["indices"]))
print("Mask:", Path(supplied["mask_file"]).name)

pcov.plot_correlation(
    covariance=selected["total"],
    covariance_ref=selected["supplied"],
    block_sizes=selected["block_sizes"],
    block_labels=selected["block_labels"],
    labels=("Computed forecast", "Supplied likelihood"),
)
pcov.plot_covariance_components(
    total=selected["total"],
    components={
        "Gaussian": selected["gaussian"],
        "SSC": selected["ssc"],
        "Connected": selected["cng"],
    },
    block_sizes=selected["block_sizes"],
    block_labels=selected["block_labels"],
    normalization="diagonal",
)

for label in ("total", "supplied"):
    modes = cov.covariance_modes(matrix=selected[label])
    print(label, "positive definite:", modes["positive_definite"],
          "minimum correlation eigenvalue:", modes["correlation_eigenvalues"][0])

## Check changes with accuracy boost

With two or more boosts, the next cell compares the *computed* matrices after
the same likelihood cuts. The highest tested boost is a comparison reference.
Generalized eigenvalues solve $C_bv=\lambda C_{m ref}v$ and bound variance
ratios for every linear combination of retained measurements. The reported
maximum departure from one is a fraction; 0.01 means 1%.

The error-bar panel selects the first source auto-correlation. It shows the
percent change of $\sqrt{C_{ii}}$ relative to the highest boost. This illustrates
why convergence needs more than diagonal agreement. Refine quadrature separately
at fixed boost, and eventually check marginalized Fisher errors and Figure of
Merit. No matrix here has its eigenvalues clipped or its diagonal repaired.

The split-triangle layout follows [Friedrich et al., Fig. 6](https://arxiv.org/abs/2012.08568).
Component maps adapt [Barreira, Krause & Schmidt, Fig. 1](https://arxiv.org/abs/1807.04266).

In [ ]:
if len(boosts) == 1:
    print("Set boosts = [1, 2] above and rerun to compare numerical accuracy.")
else:
    for boost in boosts:
        current = select_likelihood_entries(
            forecast=results[native_space][boost],
            supplied=supplied,
            block_sizes=block_sizes,
            block_labels=block_labels,
        )
        change = cov.compare_covariances(
            matrix=current["total"], reference=selected["total"],
        )
        print(boost, "maximum fractional variance change:",
              change["maximum_fractional_variance_change"])

    # The first observable is the first source auto-correlation, with
    # all its angular bins or Fourier bands consecutive in the full vector.
    nbin = 30
    reference = forecast["total"][:nbin, :nbin]
    curves = {}
    for boost in boosts[:-1]:
        curves[f"Boost {boost} / {boosts[-1]}"] = (
            results[native_space][boost]["total"][:nbin, :nbin]
        )
    coordinate = forecast["coordinate"]
    pcov.plot_covariance_diagonal(
        theta_arcmin=coordinate,
        covariances=curves,
        panel_labels=['$\\xi_+$'],
        covariance_ref=reference,
        coordinate_label='$\\ell$' if native_space == "fourier" else '$\\theta\\;[\\mathrm{arcmin}]$',
    )

## Inspect the C++ notebook interface

The compiled wrappers take vectors, matrices and cubes with named physical
axes. Python converts NumPy inputs into owned Armadillo arrays; notebook copies
are intentional. Whole Gaussian matrices and individual projection, halo and
response components are available for independent experiments. The help text
gives dimensions and units.

In [ ]:
help(ci.covariance_gaussian_fourier)
help(ci.covariance_project)
help(ci.covariance_halo_response)

## Save the calculation

The archives retain each physical component, ordering, physical assumptions and
resolved accuracy settings. The separate likelihood-selection archive contains
the compact matrices, original entry indices and the supplied total used in the
plot. Only files in `covariance/` are written; the supplied files in `data/`
remain unchanged. Running this cell again replaces the computed archives.

In [ ]:
for space in spaces:
    save_forecast(
        result=results[space][boosts[-1]],
        filename=project/"covariance"/f"forecast_{space}.npz",
    )
np.savez(file=project/"covariance/forecast_camb.npz", **camb_tables)
np.savez(
    file=project/"covariance/forecast_likelihood_selection.npz",
    **selected,
    dataset=str(dataset.relative_to(project)),
    accuracy_boost=boosts[-1],
)
print("Saved covariance components and likelihood selection.")